In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   🤖 AUTONOMOUS MULTI-AGENT AI CONTENT SYSTEM  v2.0            ║
# ║                                                                  ║
# ║   ScraperAgent (HuggingFace + DevPost)                          ║
# ║      → ContentAgent   (Groq / LLaMA 3.1)                       ║
# ║      → ImageAgent     (SDXL via HF InferenceClient)            ║
# ║      → VoiceAgent     (ElevenLabs premium TTS)                 ║
# ║      → VideoAgent     (MoviePy story + Motif-Video AI scenes)  ║
# ║      → YouTubeAgent   (auto-upload via YouTube Data API v3)    ║
# ╚══════════════════════════════════════════════════════════════════╝
import sys
from pathlib import Path
sys.path.append(str(Path("/kaggle/working")))
print("Path ready ✓")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Install All Dependencies                          ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, sys

pkgs = [
    "fastapi==0.111.0", "uvicorn[standard]==0.30.1", "requests==2.32.3",
    "huggingface-hub==0.24.0", "diffusers==0.30.3", "transformers==4.44.0",
    "accelerate==0.33.0", "safetensors==0.4.3",
    "pillow==10.4.0",       # pinned — avoids PIL._typing breakage on Py 3.12
    "moviepy==1.0.3", "imageio==2.34.2", "imageio-ffmpeg==0.5.1", "numpy==1.26.4",
    "pydub==0.25.1", "gTTS==2.5.1",
    "beautifulsoup4==4.12.3", "lxml==5.2.2",
    "schedule==1.2.2",
]

for pkg in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import torch
print("=" * 55)
print(f"PyTorch       : {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"VRAM          : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("=" * 55)
print("All packages installed ✓")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Configuration                                     ║
# ╚══════════════════════════════════════════════════════════════╝
import os, torch
from pathlib import Path
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
_secret_labels = [s.label for s in secrets.list_secrets()]

def _get_secret(key):
    return secrets.get_secret(key) if key in _secret_labels else ""

# ── API Keys ──────────────────────────────────────────────────────────────────
GROQ_API_KEY          = _get_secret("GROQ_API_KEY")
HF_TOKEN              = _get_secret("HF_TOKEN")
ELEVENLABS_API_KEY    = _get_secret("ELEVENLABS_API_KEY")
ELEVENLABS_VOICE_ID   = "21m00Tcm4TlvDq8ikWAM"   # Rachel — change to your preferred voice

YOUTUBE_CLIENT_ID     = _get_secret("YOUTUBE_CLIENT_ID")
YOUTUBE_CLIENT_SECRET = _get_secret("YOUTUBE_CLIENT_SECRET")
YOUTUBE_REFRESH_TOKEN = _get_secret("YOUTUBE_REFRESH_TOKEN")

os.environ["HF_TOKEN"]           = HF_TOKEN
os.environ["GROQ_API_KEY"]       = GROQ_API_KEY
os.environ["ELEVENLABS_API_KEY"] = ELEVENLABS_API_KEY

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR   = Path("/kaggle/working/outputs")
IMAGES_DIR = BASE_DIR / "images"
AUDIO_DIR  = BASE_DIR / "audio"
VIDEO_DIR  = BASE_DIR / "video"
LOGS_DIR   = BASE_DIR / "logs"
for _d in [IMAGES_DIR, AUDIO_DIR, VIDEO_DIR, LOGS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

# ── Groq / LLM ────────────────────────────────────────────────────────────────
GROQ_API_URL    = "https://api.groq.com/openai/v1/chat/completions"
LLM_MODEL       = "llama-3.1-8b-instant"
LLM_TEMPERATURE = 0.7
LLM_MAX_TOKENS  = 1024

# ── Image: SDXL via HF InferenceClient ───────────────────────────────────────
SDXL_MODEL_ID   = "stabilityai/stable-diffusion-xl-base-1.0"
SD_DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
SD_DTYPE        = torch.bfloat16 if SD_DEVICE == "cuda" else torch.float32
SD_IMAGE_WIDTH  = 1024
SD_IMAGE_HEIGHT = 1024
NUM_IMAGES      = 5

# ── Voice: ElevenLabs ─────────────────────────────────────────────────────────
ELEVENLABS_TTS_URL = f"https://api.elevenlabs.io/v1/text-to-speech/{ELEVENLABS_VOICE_ID}"
ELEVENLABS_MODEL   = "eleven_multilingual_v2"

# ── Video: Motif-Video-2B (AI key scenes) + MoviePy (story flow) ──────────────
MOTIF_MODEL_ID   = "Motif-Technologies/Motif-Video-2B"
VIDEO_FPS        = 8
VIDEO_FRAMES     = 16
AI_SCENE_INDICES = {0, 2, 4}   # scenes 1, 3, 5 get Motif-Video AI animation

# ── Scraper ───────────────────────────────────────────────────────────────────
HF_SCRAPE_LIMIT      = 10
DEVPOST_SCRAPE_LIMIT = 10

# ── Retry ─────────────────────────────────────────────────────────────────────
MAX_RETRIES = 3
RETRY_DELAY = 2.0

print(f"Device         : {SD_DEVICE}")
print(f"Image model    : {SDXL_MODEL_ID}")
print(f"Voice model    : ElevenLabs {ELEVENLABS_MODEL}  (voice={ELEVENLABS_VOICE_ID})")
print(f"Video strategy : Motif-Video-2B (scenes {sorted(AI_SCENE_INDICES)}) + MoviePy")
print(f"LLM            : {LLM_MODEL} via Groq")
print(f"YouTube upload : {'Enabled' if YOUTUBE_REFRESH_TOKEN else 'Disabled (no token)'}")
print("Config ready ✓")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Logger                                            ║
# ╚══════════════════════════════════════════════════════════════╝
import logging, sys
from logging.handlers import RotatingFileHandler

def get_logger(name: str) -> logging.Logger:
    logger = logging.getLogger(name)
    if logger.handlers:
        return logger
    logger.setLevel(logging.DEBUG)
    fmt = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)-20s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )
    ch = logging.StreamHandler(sys.stdout)
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)
    fh = RotatingFileHandler(LOGS_DIR / "pipeline.log",
                             maxBytes=5*1024*1024, backupCount=3, encoding="utf-8")
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(fmt)
    logger.addHandler(ch)
    logger.addHandler(fh)
    return logger

print("Logger ready ✓")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Retry Decorator                                   ║
# ╚══════════════════════════════════════════════════════════════╝
import time, functools
from typing import Tuple, Type

_retry_log = get_logger("Retry")

def with_retry(max_retries=MAX_RETRIES, delay=RETRY_DELAY,
               backoff=2.0, exceptions=(Exception,)):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            wait = delay
            for attempt in range(max_retries + 1):
                try:
                    return fn(*args, **kwargs)
                except exceptions as exc:
                    if attempt == max_retries:
                        raise
                    _retry_log.warning(
                        "Attempt %d/%d failed for %r: %s — retrying in %.0fs…",
                        attempt+1, max_retries, fn.__qualname__, exc, wait)
                    time.sleep(wait)
                    wait *= backoff
        return wrapper
    return decorator

print("Retry decorator ready ✓")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — ScraperAgent (HuggingFace + DevPost)              ║
# ╚══════════════════════════════════════════════════════════════╝
import re, time
from dataclasses import dataclass, field, asdict
from datetime import datetime
from typing import List
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

_log_sc = get_logger("ScraperAgent")
_HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"),
    "Accept-Language": "en-US,en;q=0.9",
}

@dataclass
class Topic:
    title:      str
    source:     str
    url:        str
    summary:    str       = ""
    tags:       List[str] = field(default_factory=list)
    score:      float     = 0.0
    scraped_at: str       = field(default_factory=lambda: datetime.utcnow().isoformat())

    def to_dict(self): return asdict(self)

    def as_pipeline_topic(self) -> str:
        tag_str = ", ".join(self.tags[:5])
        return (f"{self.title}" + (f" ({tag_str})" if tag_str else "")
                + (f" — {self.summary[:120]}" if self.summary else ""))


class _HFScraper:
    BASE       = "https://huggingface.co"
    PAPERS_URL = "https://huggingface.co/papers"
    SPACES_URL = "https://huggingface.co/spaces?sort=trending"

    @with_retry(max_retries=3, delay=3, exceptions=(requests.RequestException,))
    def _fetch(self, url):
        r = requests.get(url, headers=_HEADERS, timeout=20)
        r.raise_for_status()
        return BeautifulSoup(r.text, "html.parser")

    def _papers(self, limit):
        topics, seen = [], set()
        try:
            soup  = self._fetch(self.PAPERS_URL)
            links = soup.find_all("a", href=re.compile(r"^/papers/\d{4}\.\d+"))
            for link in links:
                href = link.get("href", "")
                if href in seen: continue
                seen.add(href)
                title = link.get_text(strip=True)
                if not title or len(title) < 10: continue
                parent  = link.find_parent(["article", "div", "li"])
                summary = ""
                if parent:
                    for p in parent.find_all("p"):
                        t = p.get_text(strip=True)
                        if len(t) > 40: summary = t[:200]; break
                topics.append(Topic(title=title[:120], source="huggingface",
                                    url=urljoin(self.BASE, href), summary=summary,
                                    tags=["AI","ML","Research","Paper"], score=0.9))
                if len(topics) >= limit: break
        except Exception as exc:
            _log_sc.warning("HF Papers: %s", exc)
        return topics[:limit]

    def _spaces(self, limit):
        topics = []
        try:
            soup  = self._fetch(self.SPACES_URL)
            links = soup.find_all("a", href=re.compile(r"^/spaces/[^/]+/[^/]+$"))
            for link in links[:limit]:
                title = link.get_text(strip=True) or link["href"].split("/")[-1]
                topics.append(Topic(title=title[:120], source="huggingface",
                                    url=urljoin(self.BASE, link["href"]),
                                    summary="Trending HuggingFace Space",
                                    tags=["AI","ML","Space"], score=0.8))
        except Exception as exc:
            _log_sc.warning("HF Spaces: %s", exc)
        return topics[:limit]

    def scrape(self, limit=10):
        half   = max(1, limit // 2)
        result = self._papers(half) + self._spaces(limit - half)
        _log_sc.info("HuggingFace: %d topics.", len(result))
        return result[:limit]


class _DevPostScraper:
    SEARCH_URL  = "https://devpost.com/software/search"
    _AI_QUERIES = ["artificial intelligence", "machine learning", "LLM", "generative AI"]

    @with_retry(max_retries=3, delay=3, exceptions=(requests.RequestException,))
    def _fetch(self, url, params=None):
        r = requests.get(url, headers=_HEADERS, params=params or {}, timeout=20)
        r.raise_for_status()
        return BeautifulSoup(r.text, "html.parser")

    def _parse(self, soup):
        projects = []
        cards = (soup.select("div.software-entry") or soup.select("div[id^='software-']")
                 or soup.select("article.software") or soup.select("div.gallery-item"))
        for card in cards:
            title_el = card.find("h5") or card.find("h3") or card.find("a")
            title = title_el.get_text(strip=True) if title_el else ""
            if not title: continue
            link = card.find("a", href=re.compile(r"devpost\.com/software/|^/software/"))
            url  = ""
            if link:
                href = link.get("href","")
                url  = href if href.startswith("http") else f"https://devpost.com{href}"
            tl = card.find("p", class_=re.compile(r"tagline|description"))
            summary = tl.get_text(strip=True)[:200] if tl else ""
            tags = list(dict.fromkeys(
                t.get_text(strip=True)
                for t in card.find_all("span", class_=re.compile(r"tag|tech|built"))
            ))[:8]
            projects.append({"title": title[:120], "url": url,
                             "summary": summary, "tags": tags or ["AI","DevPost"]})
        return projects

    def scrape(self, limit=10):
        topics, per_q = [], max(2, limit // len(self._AI_QUERIES))
        for query in self._AI_QUERIES:
            _log_sc.info("DevPost: searching '%s'…", query)
            try:
                for p in self._parse(self._fetch(self.SEARCH_URL, {"q": query}))[:per_q]:
                    if not any(x.title == p["title"] for x in topics):
                        topics.append(Topic(title=p["title"], source="devpost", url=p["url"],
                                            summary=p["summary"], tags=p["tags"], score=0.75))
            except Exception as exc:
                _log_sc.warning("DevPost '%s': %s", query, exc)
            if len(topics) >= limit: break
            time.sleep(1)
        _log_sc.info("DevPost: %d topics.", len(topics))
        return topics[:limit]


class ScraperAgent:
    def __init__(self):
        self._hf = _HFScraper()
        self._dp = _DevPostScraper()

    def run(self, hf_limit=HF_SCRAPE_LIMIT, devpost_limit=DEVPOST_SCRAPE_LIMIT):
        _log_sc.info("ScraperAgent: discovering topics (HF + DevPost)…")
        hf_t  = self._hf.scrape(hf_limit)
        dp_t  = self._dp.scrape(devpost_limit)
        all_t = hf_t + dp_t
        all_t.sort(key=lambda t: (-t.score, len(t.title)))
        _log_sc.info("ScraperAgent OK — %d topics (HF: %d, DevPost: %d)",
                     len(all_t), len(hf_t), len(dp_t))
        return all_t

print("ScraperAgent (HuggingFace + DevPost) ready ✓")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 — ContentAgent (Groq / LLaMA 3.1)                  ║
# ╚══════════════════════════════════════════════════════════════╝
import json, re, requests
from typing import Dict, Any

_log_ca = get_logger("ContentAgent")
_SYS = ("You are an expert YouTube content creator specialising in AI and Computer Science. "
        "Always respond with ONLY a valid JSON object — no markdown fences, no extra text.")

def _build_prompt(topic):
    return (
        f'Create a YouTube video content package for: "{topic}". '
        "Return ONLY a JSON object with keys: "
        "title (str ≤70 chars), description (str 150 words SEO), "
        "script (str 350-450 words no newlines), tags (10 strings), "
        "scenes (5 objects: scene_number, heading, description, duration_seconds), "
        "image_prompts (5 SDXL prompts: cinematic tech-themed with lighting/mood/angle), "
        "voice_script (str 300-350 words narration no newlines), "
        "thumbnail_prompt (1 SDXL prompt for eye-catching thumbnail). "
        "All strings must be single-line with no embedded newlines."
    )

class ContentAgent:
    @with_retry(max_retries=3, delay=5,
                exceptions=(requests.RequestException, ValueError, json.JSONDecodeError))
    def run(self, topic: str) -> Dict[str, Any]:
        _log_ca.info("ContentAgent: Groq API for '%s'…", topic[:80])
        resp = requests.post(
            GROQ_API_URL,
            headers={"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type":"application/json"},
            json={"model": LLM_MODEL, "temperature": LLM_TEMPERATURE, "max_tokens": LLM_MAX_TOKENS,
                  "messages": [{"role":"system","content":_SYS},
                                {"role":"user","content":_build_prompt(topic)}]},
            timeout=60,
        )
        resp.raise_for_status()
        raw = resp.json()["choices"][0]["message"]["content"]
        raw = re.sub(r"^```[a-zA-Z]*\n?","", raw.strip())
        raw = re.sub(r"```$","", raw.strip()).strip()
        raw = re.sub(r"[\x00-\x1f\x7f]"," ", raw)
        result  = json.loads(raw)
        missing = {"title","description","script","scenes","image_prompts","voice_script"} - result.keys()
        if missing: raise ValueError(f"Missing keys: {missing}")
        _log_ca.info("ContentAgent OK — title='%s'", result["title"])
        return result

print("ContentAgent ready ✓")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — ImageAgent (SDXL via HF InferenceClient)         ║
# ╚══════════════════════════════════════════════════════════════╝
from pathlib import Path
from typing import List
from huggingface_hub import InferenceClient, login

_log_ia = get_logger("ImageAgent")
_COLOURS = ["#1a1a2e","#16213e","#0f3460","#533483","#2b2d42"]

def _placeholder(prompt, idx):
    from PIL import Image, ImageDraw
    out  = IMAGES_DIR / f"scene_{idx+1:02d}.png"
    img  = Image.new("RGB", (SD_IMAGE_WIDTH, SD_IMAGE_HEIGHT), _COLOURS[idx % len(_COLOURS)])
    draw = ImageDraw.Draw(img)
    words, lines, line = prompt.split(), [], ""
    for w in words:
        if len(line+w) < 40: line += w + " "
        else: lines.append(line.strip()); line = w + " "
    lines.append(line.strip())
    y = SD_IMAGE_HEIGHT//2 - len(lines)*14
    for ln in lines[:8]: draw.text((30, y), ln, fill="white"); y += 26
    draw.text((30, SD_IMAGE_HEIGHT-40), f"Scene {idx+1}", fill="#aaaaaa")
    img.save(str(out))
    return out

class ImageAgent:
    def __init__(self):
        login(token=HF_TOKEN, add_to_git_credential=False)
        self._client = InferenceClient(token=HF_TOKEN)

    @with_retry(max_retries=2, delay=8, exceptions=(Exception,))
    def _gen(self, prompt, idx):
        out = IMAGES_DIR / f"scene_{idx+1:02d}.png"
        _log_ia.info("SDXL generating image %d/%d…", idx+1, NUM_IMAGES)
        img = self._client.text_to_image(
            f"{prompt}, cinematic lighting, 8k ultra-detailed, photorealistic, sharp focus",
            model=SDXL_MODEL_ID,
        )
        img.save(str(out))
        _log_ia.info("Image %d OK  %d KB", idx+1, out.stat().st_size//1024)
        return out

    def run(self, prompts: List[str]) -> List[Path]:
        paths = []
        for idx, prompt in enumerate(prompts[:NUM_IMAGES]):
            try: paths.append(self._gen(prompt, idx))
            except Exception as exc:
                _log_ia.warning("Image %d failed: %s — placeholder.", idx+1, exc)
                paths.append(_placeholder(prompt, idx))
        _log_ia.info("ImageAgent OK — %d images.", len(paths))
        return paths

print("ImageAgent (SDXL via HF InferenceClient) ready ✓")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8 — VoiceAgent (ElevenLabs → gTTS → silent WAV)      ║
# ╚══════════════════════════════════════════════════════════════╝
import struct, wave
from pathlib import Path
from typing import Optional
import requests

_log_va = get_logger("VoiceAgent")

class VoiceAgent:
    OUTPUT_FILE = AUDIO_DIR / "narration.mp3"

    @with_retry(max_retries=2, delay=5, exceptions=(requests.RequestException, IOError))
    def _elevenlabs(self, text):
        _log_va.info("ElevenLabs: synthesising (voice=%s)…", ELEVENLABS_VOICE_ID)
        resp = requests.post(
            ELEVENLABS_TTS_URL,
            headers={"xi-api-key": ELEVENLABS_API_KEY,
                     "Content-Type":"application/json", "Accept":"audio/mpeg"},
            json={"text": text, "model_id": ELEVENLABS_MODEL,
                  "voice_settings": {"stability":0.5,"similarity_boost":0.75,
                                     "style":0.3,"use_speaker_boost":True}},
            timeout=120,
        )
        resp.raise_for_status()
        self.OUTPUT_FILE.write_bytes(resp.content)
        _log_va.info("ElevenLabs OK — %d KB", self.OUTPUT_FILE.stat().st_size//1024)
        return self.OUTPUT_FILE

    def _gtts(self, text):
        try:
            from gtts import gTTS
            _log_va.warning("ElevenLabs unavailable — falling back to gTTS…")
            gTTS(text=text, lang="en", slow=False).save(str(self.OUTPUT_FILE))
            _log_va.info("gTTS fallback OK")
            return self.OUTPUT_FILE
        except Exception as exc:
            _log_va.warning("gTTS failed: %s", exc); return None

    def _silent(self):
        p = AUDIO_DIR / "narration.wav"
        sr, n = 22050, 22050*30
        with wave.open(str(p),"w") as wf:
            wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(sr)
            wf.writeframes(struct.pack("<"+"h"*n,*([0]*n)))
        _log_va.warning("Silent WAV fallback"); return p

    def run(self, script):
        try:
            r = self._elevenlabs(script)
            if r.exists() and r.stat().st_size > 0: return r
        except Exception as exc:
            _log_va.warning("ElevenLabs failed: %s", exc)
        r = self._gtts(script)
        if r and r.exists() and r.stat().st_size > 0: return r
        return self._silent()

print("VoiceAgent (ElevenLabs → gTTS → silent WAV) ready ✓")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 9 — VideoAgent (MoviePy story + Motif-Video AI scenes)║
# ╚══════════════════════════════════════════════════════════════╝
import gc, torch
from pathlib import Path
from typing import List, Optional

_log_vv = get_logger("VideoAgent")

class VideoAgent:
    """
    Hybrid video builder:
      AI_SCENE_INDICES  → Motif-Video-2B animation
      All other scenes  → MoviePy static image clip
    Falls back to pure slideshow if Motif-Video fails.
    """
    OUTPUT_FILE = VIDEO_DIR / "final_video.mp4"

    def _motif_clip(self, prompt, idx):
        try:
            import numpy as np
            from diffusers import DiffusionPipeline
            from moviepy.editor import ImageSequenceClip
            _log_vv.info("Motif-Video-2B: animating scene %d…", idx+1)
            pipe   = DiffusionPipeline.from_pretrained(
                MOTIF_MODEL_ID, torch_dtype=SD_DTYPE, device_map=SD_DEVICE)
            output = pipe(f"{prompt}, smooth cinematic motion, professional video, 4k")
            frames = [np.array(f) for f in output.images[0]]
            clip   = ImageSequenceClip(frames, fps=VIDEO_FPS)
            del pipe; gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            return clip
        except Exception as exc:
            _log_vv.warning("Motif-Video scene %d failed: %s", idx+1, exc)
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            return None

    @staticmethod
    def _img_clip(path, dur=5):
        from moviepy.editor import ImageClip
        return ImageClip(str(path)).set_duration(dur).resize((1280,720)).set_fps(24)

    def _hybrid(self, image_paths, audio_path, content):
        try:
            from moviepy.editor import AudioFileClip, concatenate_videoclips
            scenes      = (content or {}).get("scenes", [])
            img_prompts = (content or {}).get("image_prompts", [])
            clips       = []
            for idx, img_path in enumerate(image_paths):
                dur    = int((scenes[idx] if idx < len(scenes) else {}).get("duration_seconds",5))
                prompt = img_prompts[idx] if idx < len(img_prompts) else "AI technology"
                if idx in AI_SCENE_INDICES:
                    ai = self._motif_clip(prompt, idx)
                    if ai:
                        if ai.duration < dur:
                            loops = int(dur // ai.duration) + 1
                            ai = concatenate_videoclips([ai]*loops).subclip(0, dur)
                        else:
                            ai = ai.subclip(0, dur)
                        clips.append(ai)
                        _log_vv.info("Scene %d: Motif-Video AI ✓", idx+1)
                        continue
                clips.append(self._img_clip(img_path, dur))
                _log_vv.info("Scene %d: MoviePy image clip ✓", idx+1)
            if not clips: return None
            video = concatenate_videoclips(clips, method="compose")
            if audio_path and Path(str(audio_path)).exists() and Path(str(audio_path)).stat().st_size > 0:
                audio = AudioFileClip(str(audio_path))
                if video.duration < audio.duration:
                    loops = int(audio.duration // video.duration) + 1
                    video = concatenate_videoclips([video]*loops).subclip(0, audio.duration)
                else:
                    video = video.subclip(0, audio.duration)
                video = video.set_audio(audio)
            _log_vv.info("Rendering final video (%.1fs)…", video.duration)
            video.write_videofile(str(self.OUTPUT_FILE), fps=VIDEO_FPS, codec="libx264",
                                  audio_codec="aac",
                                  temp_audiofile=str(VIDEO_DIR/"tmp_audio.m4a"),
                                  remove_temp=True, logger=None)
            _log_vv.info("VideoAgent OK → %s", self.OUTPUT_FILE)
            return self.OUTPUT_FILE
        except Exception as exc:
            _log_vv.error("Hybrid build failed: %s", exc); return None

    def _slideshow(self, image_paths, audio_path):
        try:
            from moviepy.editor import AudioFileClip, ImageClip, concatenate_videoclips
            _log_vv.warning("Pure slideshow fallback…")
            clips = [ImageClip(str(p)).set_duration(5).resize((1280,720)).set_fps(24)
                     for p in image_paths]
            video = concatenate_videoclips(clips, method="compose")
            if audio_path and Path(str(audio_path)).exists():
                audio = AudioFileClip(str(audio_path))
                video = video.subclip(0, min(video.duration,audio.duration)).set_audio(audio)
            video.write_videofile(str(self.OUTPUT_FILE), fps=24,
                                  codec="libx264", audio_codec="aac", logger=None)
            return self.OUTPUT_FILE
        except Exception as exc:
            _log_vv.error("Slideshow failed: %s", exc); return None

    def run(self, image_paths, audio_path, title, content=None):
        result = self._hybrid(image_paths, audio_path, content)
        if result and result.exists(): return result
        return self._slideshow(image_paths, audio_path)

print("VideoAgent (MoviePy + Motif-Video-2B hybrid) ready ✓")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 10 — YouTubeAgent (YouTube Data API v3)               ║
# ╚══════════════════════════════════════════════════════════════╝
import requests
from pathlib import Path
from typing import Optional, List

_log_yt = get_logger("YouTubeAgent")
_TOKEN_URL  = "https://oauth2.googleapis.com/token"
_UPLOAD_URL = "https://www.googleapis.com/upload/youtube/v3/videos"
_CHUNK_SIZE = 8 * 1024 * 1024   # 8 MB

class YouTubeAgent:
    def _token(self):
        _log_yt.info("Refreshing YouTube access token…")
        resp = requests.post(_TOKEN_URL, data={
            "client_id": YOUTUBE_CLIENT_ID, "client_secret": YOUTUBE_CLIENT_SECRET,
            "refresh_token": YOUTUBE_REFRESH_TOKEN, "grant_type": "refresh_token",
        }, timeout=30)
        resp.raise_for_status()
        t = resp.json().get("access_token","")
        if not t: raise RuntimeError(f"No access_token: {resp.text[:200]}")
        return t

    def _start_session(self, token, meta, size):
        resp = requests.post(_UPLOAD_URL,
            headers={"Authorization": f"Bearer {token}",
                     "Content-Type": "application/json; charset=UTF-8",
                     "X-Upload-Content-Type": "video/mp4",
                     "X-Upload-Content-Length": str(size)},
            params={"uploadType":"resumable","part":"snippet,status"},
            json=meta, timeout=30)
        resp.raise_for_status()
        url = resp.headers.get("Location","")
        if not url: raise RuntimeError("No upload URL from YouTube")
        return url

    def _upload(self, upload_url, path):
        size, uploaded = path.stat().st_size, 0
        with open(path,"rb") as fh:
            while uploaded < size:
                chunk = fh.read(_CHUNK_SIZE)
                end   = uploaded + len(chunk) - 1
                resp  = requests.put(upload_url,
                    headers={"Content-Range": f"bytes {uploaded}-{end}/{size}",
                             "Content-Type": "video/mp4"},
                    data=chunk, timeout=120)
                if resp.status_code in (200,201):
                    _log_yt.info("Upload 100% ✓"); return resp.json()
                if resp.status_code == 308:
                    r = resp.headers.get("Range", f"bytes=0-{end}")
                    uploaded = int(r.split("-")[1]) + 1
                    _log_yt.info("Uploaded %.1f%%…", uploaded/size*100); continue
                resp.raise_for_status()
        raise RuntimeError("Upload incomplete")

    def upload(self, video_path, title, description="", tags=None,
               category_id="28", privacy="public"):
        if not video_path.exists(): raise FileNotFoundError(str(video_path))
        size = video_path.stat().st_size
        _log_yt.info("Uploading '%s' (%.1f MB, %s)…", title[:60], size/1e6, privacy)
        token = self._token()
        meta  = {
            "snippet": {"title": title[:100], "description": description[:5000],
                        "tags": (tags or [])[:500], "categoryId": category_id},
            "status":  {"privacyStatus": privacy, "selfDeclaredMadeForKids": False},
        }
        url = self._upload(self._start_session(token, meta, size), video_path)
        vid = url.get("id","")
        yt  = f"https://www.youtube.com/watch?v={vid}"
        _log_yt.info("YouTubeAgent OK — %s", yt)
        return yt

print("YouTubeAgent (YouTube Data API v3) ready ✓")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 11 — Orchestrator (Autonomous Multi-Agent Coordinator) ║
# ╚══════════════════════════════════════════════════════════════╝
import gc, json, time, uuid, torch
from dataclasses import dataclass, field, asdict
from typing import Optional, Dict, Any, List
from pathlib import Path

_log_orch = get_logger("Orchestrator")

@dataclass
class PipelineState:
    job_id:         str
    topic:          str                      = ""
    status:         str                      = "pending"
    content:        Optional[Dict[str,Any]]  = None
    image_paths:    List[str]                = field(default_factory=list)
    audio_path:     Optional[str]            = None
    video_path:     Optional[str]            = None
    youtube_url:    Optional[str]            = None
    scraped_topics: List[Dict]               = field(default_factory=list)
    errors:         List[str]                = field(default_factory=list)
    started_at:     float                    = field(default_factory=time.time)
    elapsed_s:      float                    = 0.0

    def to_dict(self): return asdict(self)

    def snapshot(self):
        p = LOGS_DIR / f"{self.job_id}_state.json"
        with open(p,"w",encoding="utf-8") as fp:
            json.dump(self.to_dict(), fp, indent=2, default=str)


class Orchestrator:
    def __init__(self, job_id=None):
        self.state = PipelineState(job_id=job_id or str(uuid.uuid4())[:8])
        _log_orch.info("Orchestrator ready (job_id=%s)", self.state.job_id)

    def _err(self, stage, exc):
        msg = f"[{stage}] {type(exc).__name__}: {exc}"
        _log_orch.error(msg); self.state.errors.append(msg)

    def _free_gpu(self):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            free = (torch.cuda.get_device_properties(0).total_memory
                    - torch.cuda.memory_reserved(0))
            _log_orch.info("GPU cleared. Free VRAM: %.1f GB", free/1e9)

    # ── Stage 0: Scrape ───────────────────────────────────────────────────────
    def _scrape(self):
        _log_orch.info("▶ Stage 0/5 — ScraperAgent (HuggingFace + DevPost)")
        try:
            topics = ScraperAgent().run()
            self.state.scraped_topics = [t.to_dict() for t in topics]
            if not topics: return None
            best = topics[0].as_pipeline_topic()
            _log_orch.info("ScraperAgent OK — best='%s'", best[:80])
            return best
        except Exception as exc:
            self._err("ScraperAgent", exc); return None

    # ── Stage 1: Content ──────────────────────────────────────────────────────
    def _content(self, topic):
        _log_orch.info("▶ Stage 1/5 — ContentAgent (Groq / LLaMA 3.1)")
        try:
            self.state.content = ContentAgent().run(topic)
            self.state.topic   = topic
            _log_orch.info("ContentAgent OK — '%s'", self.state.content["title"])
            return True
        except Exception as exc:
            self._err("ContentAgent", exc); return False

    # ── Stage 2: Images ───────────────────────────────────────────────────────
    def _images(self):
        _log_orch.info("▶ Stage 2/5 — ImageAgent (SDXL)")
        if not self.state.content:
            _log_orch.error("Skipping — no content."); return False
        try:
            paths = ImageAgent().run(self.state.content.get("image_prompts",[]))
            self.state.image_paths = [str(p) for p in paths]
            self._free_gpu()
            _log_orch.info("ImageAgent OK — %d images", len(paths))
            return bool(paths)
        except Exception as exc:
            self._err("ImageAgent", exc); self._free_gpu(); return False

    # ── Stage 3: Voice ────────────────────────────────────────────────────────
    def _voice(self):
        _log_orch.info("▶ Stage 3/5 — VoiceAgent (ElevenLabs)")
        if not self.state.content:
            _log_orch.error("Skipping — no content."); return False
        try:
            script = (self.state.content.get("voice_script")
                      or self.state.content.get("script",""))
            path   = VoiceAgent().run(script)
            self.state.audio_path = str(path)
            _log_orch.info("VoiceAgent OK — %s", path.name)
            return True
        except Exception as exc:
            self._err("VoiceAgent", exc); return False

    # ── Stage 4: Video ────────────────────────────────────────────────────────
    def _video(self):
        _log_orch.info("▶ Stage 4/5 — VideoAgent (MoviePy + Motif-Video)")
        if not self.state.image_paths:
            _log_orch.error("Skipping — no images."); return False
        try:
            path = VideoAgent().run(
                image_paths = [Path(p) for p in self.state.image_paths],
                audio_path  = Path(self.state.audio_path) if self.state.audio_path else None,
                title       = (self.state.content or {}).get("title","AI Video"),
                content     = self.state.content,
            )
            if path:
                self.state.video_path = str(path)
                _log_orch.info("VideoAgent OK — %s", path.name)
                return True
            return False
        except Exception as exc:
            self._err("VideoAgent", exc); return False

    # ── Stage 5: YouTube ──────────────────────────────────────────────────────
    def _youtube(self):
        _log_orch.info("▶ Stage 5/5 — YouTubeAgent")
        if not self.state.video_path:
            _log_orch.error("Skipping — no video."); return False
        if not YOUTUBE_REFRESH_TOKEN:
            _log_orch.warning("No YOUTUBE_REFRESH_TOKEN — skipping upload."); return False
        try:
            c   = self.state.content or {}
            url = YouTubeAgent().upload(
                video_path  = Path(self.state.video_path),
                title       = c.get("title", self.state.topic[:100]),
                description = c.get("description",""),
                tags        = c.get("tags",[]),
                privacy     = "public",
            )
            self.state.youtube_url = url
            _log_orch.info("YouTubeAgent OK — %s", url)
            return True
        except Exception as exc:
            self._err("YouTubeAgent", exc); return False

    # ── Main run ──────────────────────────────────────────────────────────────
    def run(self, topic=None, upload_to_youtube=True):
        self.state.status = "running"
        _log_orch.info("═"*60)
        _log_orch.info("  PIPELINE START  job_id=%s", self.state.job_id)
        _log_orch.info("═"*60)

        if not topic:
            topic = self._scrape()
        if not topic:
            topic = "Latest AI and Machine Learning Breakthroughs 2025"
            _log_orch.warning("Fallback topic: '%s'", topic)

        self.state.snapshot()
        self._content(topic);  self.state.snapshot()
        self._images();        self.state.snapshot()
        self._voice();         self.state.snapshot()
        self._video();         self.state.snapshot()
        if upload_to_youtube:
            self._youtube();   self.state.snapshot()

        self.state.elapsed_s = round(time.time() - self.state.started_at, 1)
        self.state.status    = "error" if self.state.errors else "done"
        _log_orch.info("═"*60)
        if self.state.errors:
            _log_orch.warning("PIPELINE DONE with %d error(s) in %ss",
                              len(self.state.errors), self.state.elapsed_s)
        else:
            _log_orch.info("PIPELINE COMPLETE ✓ in %ss", self.state.elapsed_s)
        _log_orch.info("═"*60)
        self.state.snapshot()
        return self.state

print("Orchestrator ready ✓")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 12 — RUN AUTONOMOUS PIPELINE                          ║
# ╚══════════════════════════════════════════════════════════════╝
#
#  OPTIONS:
#    topic=None           → ScraperAgent auto-picks trending topic (HF + DevPost)
#    topic="Your topic"  → use a specific manual topic
#    upload_to_youtube    → True to publish to YouTube, False to skip
#
# ─────────────────────────────────────────────────────────────────────────────

orch  = Orchestrator()
state = orch.run(
    topic             = None,    # ← None = fully autonomous
    upload_to_youtube = True,    # ← False to skip upload
)

# ─── Summary ──────────────────────────────────────────────────────────────────
d = state.to_dict()
print()
print("═" * 65)
print("  PIPELINE RESULT SUMMARY")
print("═" * 65)
print(f"  Job ID    : {d['job_id']}")
print(f"  Status    : {d['status']}")
print(f"  Topic     : {d['topic']}")
print(f"  Title     : {(d.get('content') or {}).get('title', '—')}")
print(f"  Images    : {len(d['image_paths'])} file(s)  [SDXL InferenceClient]")
print(f"  Audio     : {d['audio_path'] or '—'}  [ElevenLabs]")
print(f"  Video     : {d['video_path'] or '—'}  [MoviePy + Motif-Video hybrid]")
print(f"  YouTube   : {d['youtube_url'] or '—'}")
print(f"  Elapsed   : {d['elapsed_s']}s")
print(f"  Errors    : {len(d['errors'])}")
for err in d["errors"]:
    print(f"    ✗ {err}")
print("═" * 65)

if d.get("content"):
    c = d["content"]
    print(f"\n📄 Title          : {c['title']}")
    print(f"\n🏷️  Tags            : {c.get('tags', [])}")
    print(f"\n📝 Script preview  :\n   {c['script'][:300]}…")
    print(f"\n🎨 Image prompts:")
    for i, p in enumerate(c.get("image_prompts",[]),1):
        print(f"  {i}. {p[:100]}")
    print(f"\n🖼️  Image files:")
    for p in d["image_paths"]: print(f"   {p}")
    print(f"\n🎬 Video  : {d['video_path'] or 'Not generated'}")
    print(f"\n📺 YouTube: {d['youtube_url'] or '—'}")

print(f"\n📡 Scraped topics ({len(d['scraped_topics'])}):")
for t in d["scraped_topics"][:5]:
    print(f"  [{t['source']:12s}] {t['title'][:70]}")

import torch
if torch.cuda.is_available():
    print(f"\nGPU Memory Used  : {torch.cuda.memory_allocated(0)/1e9:.2f} GB")
    print(f"GPU Memory Total : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")